In [0]:
%sql
-- Give all users catalog and schema permissons.

-- CREATE SCHEMA IF NOT EXISTS workspace.medallion_data;
-- GRANT USE CATALOG ON CATALOG workspace TO `account users`;
-- GRANT USE SCHEMA, CREATE TABLE, SELECT, MODIFY ON SCHEMA workspace.medallion_data TO `account users`;

In [0]:
from pyspark.sql.types import StructType, StructField, DoubleType, StringType

In [0]:
# Data in s3 bucket is header-less
schema = StructType([
    StructField("fLength", DoubleType(), True),
    StructField("fWidth", DoubleType(), True),
    StructField("fSize", DoubleType(), True),
    StructField("fConc", DoubleType(), True),
    StructField("fConc1", DoubleType(), True),
    StructField("fAsym", DoubleType(), True),
    StructField("fM3Long", DoubleType(), True),
    StructField("fM3Trans", DoubleType(), True),
    StructField("fAlpha", DoubleType(), True),
    StructField("fDist", DoubleType(), True),
    StructField("class", StringType(), True)
])

In [0]:
# s3 paths
raw_data_path = "s3://ncf-2026-spring-telescope/dump/"
checkpoint_path = "s3://ncf-2026-spring-telescope/_checkpoints/bronze_telescope/"

In [0]:
# Stream in raw data
df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .schema(schema) 
  .load(raw_data_path)
)

In [0]:
# Write bronze table to catalog.
(df.writeStream
  .option("checkpointLocation", checkpoint_path)
  .trigger(availableNow=True) 
  .toTable("workspace.medallion_data.bronze_telescope") 
)

In [0]:
# Check bronze table
display(spark.table("workspace.medallion_data.bronze_telescope"))